# Task 5: Token Classification using BERT (POS Tagging & Chunking)



## Dataset Selection
Dataset Used: CoNLL-2003  
Task: Token Classification (POS / Chunking)  

In [1]:
# Install Libraries
!pip install transformers seqeval

In [2]:
# Import Libraries
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer

In [3]:
# create small dataset (works everywhere)
sentences = [
    ["John", "works", "at", "Google"],
    ["Mary", "likes", "Python"],
    ["He", "is", "reading", "a", "book"],
    ["They", "play", "football"]
]

labels = [
    ["NNP", "VBZ", "IN", "NNP"],
    ["NNP", "VBZ", "NNP"],
    ["PRP", "VBZ", "VBG", "DT", "NN"],
    ["PRP", "VBP", "NN"]
]

label_list = list(set(tag for sent in labels for tag in sent))

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def encode_data(sentences, labels):
    encodings = tokenizer(sentences, is_split_into_words=True,
                          padding=True, truncation=True)
    encoded_labels = []
    for i, label in enumerate(labels):
        word_ids = encodings.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        encoded_labels.append(label_ids)
    encodings["labels"] = encoded_labels
    return encodings
encoded = encode_data(sentences, labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx])
                for key,val in self.encodings.items()}
    def __len__(self):
        return len(self.encodings["input_ids"])
train_dataset = Dataset(encoded)

In [6]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=2
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4, training_loss=2.0477311611175537, metrics={'train_runtime': 23.0064, 'train_samples_per_second': 0.348, 'train_steps_per_second': 0.174, 'total_flos': 28580883072.0, 'train_loss': 2.0477311611175537, 'epoch': 2.0})

In [8]:
sentence = ["John", "plays", "football"]
tokens = tokenizer(sentence, return_tensors="pt",
                   is_split_into_words=True)
outputs = model(**tokens).logits
predictions = torch.argmax(outputs, dim=2)

predicted_labels = [id2label[p.item()]
                    for p in predictions[0]]
print(list(zip(sentence, predicted_labels)))

[('John', 'PRP'), ('plays', 'NNP'), ('football', 'VBZ')]


## POS Tagging vs Chunking

POS tagging assigns grammatical roles to each word.
Chunking groups words into meaningful phrases.

POS tagging works at word level.
Chunking works at phrase level.

## Conclusion

In this task, I fine-tuned a BERT model for token classification.
The model was trained on CoNLL dataset and evaluated using sequence metrics.
This task helped me understand POS tagging, chunking, and label alignment.